# **Stages 7 & 8: Prototype Development (MVP) & Performance Evaluation**

Final hardened Morpheus Guard system combining the deep autoencoder with a context-aware hybrid rules engine, served via a Gradio web dashboard with real-time anomaly scoring and audit logging.

In [ ]:
# --- FINAL HARDENED MORPHEUS HYBRID MVP (Stages 7 & 8) ---

import torch
import torch.nn as nn
import gradio as gr
import re
from sentence_transformers import SentenceTransformer
import datetime

# 1. ARCHITECTURE & INIT
class SpearPhishingAE(nn.Module):
    def __init__(self, input_dim=384):
        super(SpearPhishingAE, self).__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 32))
        self.decoder = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, input_dim))
    def forward(self, x): return self.decoder(self.encoder(x))

device = torch.device("cpu")
model_ae = SpearPhishingAE().to(device)
model_ae.load_state_dict(torch.load("/content/spear_phishing_ae_weights.pth", map_location=device, weights_only=True))
model_ae.eval()
text_embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# סף אופטימלי שהגדרנו לפי ניתוח סטטיסטי
OPTIMAL_THRESHOLD = 0.0018

# 2. IMPROVED HYBRID LOGIC: CONTEXT-AWARE & ANTI-MITM
def hybrid_security_logic(email_text):
    # --- AI LAYER ---
    embedding = text_embedder.encode([str(email_text)], convert_to_tensor=True).to(device)
    with torch.no_grad():
        recon = model_ae(embedding)
        ai_score = torch.mean((embedding - recon)**2).item()

    # --- HEURISTICS LAYER (Rules) ---
    is_rule_threat = False
    reasons = []

    # Whitelist - דומיינים שלא נחסום עליהם לינקים
    trusted_domains = ['zoom.us', 'microsoft.com', 'sharepoint.com', 'nvidia.com', 'google.com', 'hit.ac.il']
    is_trusted_link = any(domain in email_text.lower() for domain in trusted_domains)

    # חוק 1: הגנה מפני פישינג (מופעל רק אם הלינק לא מוכר)
    if not is_trusted_link:
        if re.search(r'(urgent|immediately|verify|lockout|expires).*(http|www|link|click|portal)', email_text, re.IGNORECASE):
            is_rule_threat = True
            reasons.append("🎣 Phishing Intent (Untrusted Link)")

    # חוק 2: הגנה מפני הונאה פיננסית / MITM (חדש!)
    # זה יתפוס את הניסיונות לשנות פרטי בנק או העברות לאופשור
    if re.search(r'(banking partner|account changed|new account|offshore|cayman|wire transfer|routing number)', email_text, re.IGNORECASE):
        is_rule_threat = True
        reasons.append("💸 Financial Fraud / MITM Pattern")

    # חוק 3: קבצים מסוכנים (תמיד נחסום)
    if re.search(r'\.(exe|bat|js|scr|vbs)', email_text, re.IGNORECASE):
        is_rule_threat = True
        reasons.append("🚨 Dangerous Attachment Detected")

    return is_rule_threat, ai_score, reasons

# 3. UI WRAPPER WITH 3-TIER ALERTING
def scan_email_ui(email_text):
    is_blocked, score, reasons = hybrid_security_logic(email_text)
    timestamp = datetime.datetime.now().strftime("%H:%M:%S")

    # עדיפות 1: חסימה מוחלטת על בסיס חוקים
    if is_blocked:
        return "🚩 [BLOCKED]", f"ALERT ({timestamp}): Blocked by Rule Engine.\nReason: {reasons}\nAI Score: {score:.6f}"

    # עדיפות 2: אזהרה על בסיס אנומליה של ה-AI
    elif score > OPTIMAL_THRESHOLD:
        return "⚠️ [WARNING]", f"CAUTION ({timestamp}): High structural anomaly.\nAI Score: {score:.6f}\nStatus: Flagged for SOC Review."

    # עדיפות 3: אישור
    else:
        return "✅ [CLEAN]", f"LOG ({timestamp}): Matches business baseline.\nAI Score: {score:.6f}"

# 4. LAUNCH UI
with gr.Blocks(title="Morpheus Guard") as demo:
    gr.Markdown("# 🛡️ Morpheus Hybrid Guard (MVP)")
    gr.Markdown("Real-time traffic analysis combining Deep Learning & Context-Aware Rules.")
    with gr.Row():
        with gr.Column():
            input_box = gr.Textbox(label="Incoming Email Content", lines=8, placeholder="Paste email content here...")
            btn = gr.Button("SCAN TRAFFIC", variant="primary")
        with gr.Column():
            output_status = gr.Label(label="System Verdict")
            logs_box = gr.Textbox(label="Logging & Alerting Module", lines=5)
    btn.click(fn=scan_email_ui, inputs=input_box, outputs=[output_status, logs_box])

demo.launch(share=True, inline=False)